# Data preprataion for HEAL Abstract->Domain (Researh Focus) training

This notebook builds several sets of multi-label training data for HEAL Abstract Domain classification. It uses mapping files, data from [Heal Data Platform](https://healdata.org/portal) and [NIH Reporter](https://reporter.nih.gov/) websites.

For the domain labels [10 HEAL Pain Domains](https://www.nih.gov/heal/heal-initiative-requirements/data-sharing-policy/common-data-elements-cdes-program), also known as Research Focus, are used for _full datasets and 7 domains are used for _collapsed dataset.

Abstracts are retrieved from [Heal Data Platform](https://healdata.org/portal) metadata endpoint for heal_ datasets, and from [NIH Reporter](https://reporter.nih.gov/) API.

This notebook creates 6 datasets:

* heal_basic_full - 84 studies from [Heal Data Platform](https://healdata.org/portal) mapped by users to [**10** HEAL Pain Domains](https://www.nih.gov/heal/heal-initiative-requirements/data-sharing-policy/common-data-elements-cdes-program)
* heal_extended_full - 658 studies from [Heal Data Platform](https://healdata.org/portal) with keywords in spending_categories_desc mapped to [**10** HEAL Pain Domains](https://www.nih.gov/heal/heal-initiative-requirements/data-sharing-policy/common-data-elements-cdes-program)
* nih_reporter_full - 146,324 studies from [NIH Reporter](https://reporter.nih.gov/) with keywords in spending_categories_desc  mapped to [**10** HEAL Pain Domains](https://www.nih.gov/heal/heal-initiative-requirements/data-sharing-policy/common-data-elements-cdes-program)


* heal_basic_extended - 84 studies from [Heal Data Platform](https://healdata.org/portal) mapped by users to **7** HEAL Pain Domains
* heal_extended_extended - 658 studies from [Heal Data Platform](https://healdata.org/portal) with keywords in spending_categories_desc mapped to **7** HEAL Pain Domains
* nih_reporter_extended - 146,324 studies from [NIH Reporter](https://reporter.nih.gov/) with keywords in spending_categories_desc  mapped to **7** HEAL Pain Domains

Datasets format:
| hdp_id   | appl_id  | project_num           | abstract_text                                                        | domains                                |
|----------|----------|-----------------------|---------------------------------------------------------------------|----------------------------------------|
| HDP00033 | 10056337 | 1R61NS113258-01A1     | PROJECT SUMMARY. Taxanes are among the most efficacious…            | Depression, Pain Interference, Sleep, Substance Use |
| HDP00097 | 10157953 | 1R61NS118651-01A1     | PROJECT SUMMARY.  Chronic pain represents a public health…          | Anxiety, Substance Use                 |
| HDP00110 | 9870024  | 1UG3NR019196-01       | Chronic musculoskeletal pain (CMP) is the most common…              | Depression, Sleep, Substance Use       |

hdp_id - Heal Data Platform ID

appl_id - NIH Reporter application ID

project_num - NIH Reporter project number






## Libraries

In [ ]:
# Uncomment the line to install libraries used in this notebook
#!pip install -q numpy pandas jupyter openpyxl matplotlib

In [ ]:
import os
import pandas as pd
import ast
import requests
import matplotlib.pyplot as plt
import json
import csv

## Input/Output directories

In [ ]:
# Directory setup for inputs and outputs, change as needed
input_dir = "./inputs"
output_dir = "./outputs/datasets"
plots_dir = "./outputs/plots"

# Create directories if they do not exist
os.makedirs(input_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)
os.makedirs(plots_dir, exist_ok=True)

# Data preparation

## Domain Configurations

Create lists for full and collapsed HEAL pain domains (https://www.nih.gov/heal/heal-initiative-requirements/data-sharing-policy/common-data-elements-cdes-program)

In [ ]:
# Full HEAL domain set
heal_pain_domains_full = [
    "Pain Intensity",
    "Pain Interference",
    "Physical Functioning",
    "Sleep",
    "Pain Catastrophizing",
    "Depression",
    "Anxiety",
    "Global Satisfaction with Treatment",
    "Substance Use",
    "Quality of Life"
]

# Collapsed maps/dictionaries
pain_collapse_map = {
    "Pain Intensity": "Pain",
    "Pain Interference": "Pain",
    "Pain Catastrophizing": "Pain"
}
satisfaction_qol_collapse_map = {
    "Global Satisfaction with Treatment": "Quality of Life"
}

# Collapsed domain set
heal_pain_domains_collapsed = [
    "Pain",
    "Physical Functioning",
    "Sleep",
    "Depression",
    "Anxiety",
    "Substance Use",
    "Quality of Life"
]

## Benchmark Study Exclusions

Create list of studies to exclude from training, they are used for evaluation of the models

In [ ]:
exclude_ids = [
    "HDP00895",
    "HDP01476",
    "HDP01498",
    "HDP01495",
    "HDP01340",
    "HDP01329",
    "HDP01011",
    "HDP00933",
    "HDP00429"
]

Create exclusion lists for different identifier types.

- hdp_id is unique identifier in the HEAL DATA PLATFORM
- appl_id is unique identifier in the NIH REPORTER
- project_num is project identifier in the NIH REPORTER

We use combinations of these identifiers to exclude studies from training on different data prep steps:
- heal_basic dataset: hdp_id, project_num, appl_id
- heal_extended dataset: hdp_id, appl_id
- nih_reporter dataset: appl_id

In [ ]:
exclude_appl_ids = set()
exclude_project_nums = set()
exclude_id_lookup = {}

for hdp_id in exclude_ids:
    url = f"https://healdata.org/mds/metadata/{hdp_id}"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            response = r.json()
            nih_reporter = response.get("nih_reporter", {})
            appl_id = nih_reporter.get("appl_id")
            project_num = nih_reporter.get("project_num")
            exclude_id_lookup[hdp_id] = {
                "appl_id": str(appl_id) if appl_id else None,
                "project_num": str(project_num) if project_num else None
            }
            if appl_id:
                exclude_appl_ids.add(str(appl_id))
            if project_num:
                exclude_project_nums.add(str(project_num))
        else:
            print(f"{hdp_id}: request failed ({r.status_code})")
    except Exception as e:
        print(f"Error for {hdp_id}: {e}")

print("Exclude appl_ids:", exclude_appl_ids)
print("Exclude project_nums:", exclude_project_nums)
print("All exclude_id_lookup:", exclude_id_lookup)

## Mapping files

Three mapping files are used to map HEAL studies to HEAL Pain Domains:
* HEAL_MDS_Checklist_Status_20260429.csv - this file allows to identify the study records that have submitted CDEs
* HEAL CDEs to Drupal ID Mapping.csv - this file maps CDE instruments to drupal IDs, which can be used to access the VLMD file in the HEAL MDS
* HEAL CDE Core and Supplemental Questionnaires List 4.17.26_vz.xlsx - this file allows to identify the domain associated with the CDE (e.g., pain, sleep, duration, anxiety)

Place mapping files to your inputs folder, and load data in next cells

### Checklist File

In [ ]:
# Path to the CSV file
checklist_csv_path = os.path.join(input_dir, "HEAL_MDS_Checklist_Status_20260429.csv")
# Read the checklist CSV file
checklist_df = pd.read_csv(checklist_csv_path)
# Filter for rows with CDE selections (not "No" or empty) and not in the exclude list
filtered_checklist_df = checklist_df[
    (checklist_df["VLMD: CDE selections"].notna()) &
    (checklist_df["VLMD: CDE selections"] != "No") &
    (~checklist_df["HDP ID"].isin(exclude_ids)) &
    (~checklist_df["Project number"].astype(str).isin(exclude_project_nums)) 
]
print(f"Filtered DataFrame has {len(filtered_checklist_df)} rows with valid CDE selections.")
# Display the relevant columns for quick review
filtered_checklist_df[["HDP ID", "VLMD: CDE selections"]].head()


### Drupal mapping file

In [ ]:
# Path to the CSV file in the inputs folder 
cde_drupal_mapping_csv_path = os.path.join(input_dir, "HEAL CDEs to Drupal ID Mapping.csv")

# Load the CSV file, specifying drupal_id as integer
cde_drupal_mapping_df = pd.read_csv(
    cde_drupal_mapping_csv_path,
    dtype={"drupal_id": str}
)

print(f"Loaded {len(cde_drupal_mapping_df)} rows from 'HEAL CDEs to Drupal ID Mapping.csv'.")
print("drupal_id dtype:", cde_drupal_mapping_df['drupal_id'].dtype)
cde_drupal_mapping_df.head()

### Questionnaires file

In [ ]:
# Path to the Excel file
cde_questionnaires_excel_path = os.path.join(input_dir, "HEAL CDE Core and Supplemental Questionnaires List 4.17.26_vz.xlsx")

# Read the 'HEAL CDE Questionnaires' sheet into a DataFrame
cde_questionnaires_df = pd.read_excel(cde_questionnaires_excel_path, sheet_name="HEAL CDE Questionnaires", header=2)

print(f"Loaded {len(cde_questionnaires_df)} rows from 'HEAL CDE Questionnaires' sheet.")
cde_questionnaires_df.head()

Domain Mappings

## CDE Names Standartization

There is a mismatch in several CDEs names in the drupal file and checklist files, that prevents correct mapping.
The fix below allows to resolve this issue.

In [ ]:
# Update multiple CDE names in cde_drupal_mapping_df for given drupal_ids

updates = {
    "5096": "Generalized Anxiety Disorder 2- GAD 2",
    "5061": "Sleep Disturbance Short Form 6a PROMIS + Sleep Duration Question",
    "5051": "Sleep Disturbance Short Form 6a PROMIS + Sleep Duration Question",
    "5071": "Tobacco, Alcohol, Prescription medications, and other Substance 1 - TAPS 1",
    "5131": "Demographics- Pediatrics",
    "5111": "PEG Pain Screening Tool",
    "5876": "Tobacco, Alcohol, Prescription medications, and other Substance Part 2 - TAPS 2",
    "6141": "Self-Administered Rating Scale for Pubertal Development- PDS",
    "7646": "Primary Care PTSD Screen for DSM 5 - PC PTSD 5",
    "7401": "Keele STarT Back Screening Tool",
    "7651": "PTSD Checklist for DSM-5 (PCL-5)",
    "7591": "Ohio State Traumatic Brain Injury Identification - OSU TBI",
    "8916": "Migraine Disability Assessment Questionnaire - MIDAS",
    "6546": "Self-Administered Comorbidity Questionnaire - SCQ [A2CPS]",
    "9436": "Morphine Milligram Equivalent (MME) Calculation",
    "9476": "Pediatric Sleep Related Impairment 8a [PROMIS]",
    "9241": "Cancer Behavior Inventory Brief Version- CBI B",
    "9731": "Social Readjustment Rating Scale - SRRS",
    "5746": "Children's Behavior Questionnaire very short form- CBQ VSF",
    "5546": "Acculturation Rating Scale for Mexican Americans II (ARSMA II) Scale 1",
    "5551": "Acculturation Rating Scale for Mexican Americans II (ARSMA II) Scale 2",
    "8206": "Pain Self Efficacy Questionnaire *2 item - PSEQ 2",
    "8336": "Short Physical Performance Battery",
    "8221": "Pain Self Efficacy Questionnaire *10 item - PSEQ 10",
    "8286": "Roland Morris Low Back Pain and Disability Questionnaire - RMDQ",
    "5896": "Timeline Follow back Method Assessment - TLFB",
    "9686": "Multidimensional Psychological Flexibility Inventory (MPFI)",
    "9651": "Cognitive Emotion Regulation Questionnaire Positive Reappraisal (CERQ)",
    "8366": "World Health Organization Quality of Life Brief with Demographics and Administration Questions - WHOQOL BREF",
    "4956": "Brief Pain Inventory SF *7 day recall - BPI SF ",
    "7616": "Pain Visual Analogue Scale Numeric rating scale - VAS NRS",
    "4891": "Pain Catastrophizing Questionnaire Parent Report- PCS P",
    "8881": "Brief Assessment of Recovery Capital - BARC 10",
    "6031": "Coping Strategies Questionnaire Modified 24 item- CSQ 24  ",
    "8146": "Self-Efficacy for Managing Chronic Conditions Managing Symptoms SF 8a [PROMIS]",
    "6251": "Fear Avoidance Beliefs Questionnaire- FABQ",
    "7196": "Central Sensitization Inventory - CSI",
    "9531": "Telehealth Usability Questionnaire - TUQ",
    "9451": "World Health Organization Quality of Life 2 Item - WHOQOL *2 Item",
    "9496": "Sickle Cell Disease Self Efficacy Scale",
    "9456": "Patient Health Questionnaire for Adolescents - PHQ M | PHQ A",
    "5376": "Patient Health Questionnaire 9 - PHQ 9 ",
    "5361": "Patient Health Questionnaire 8 - PHQ 8"
    
}


for drupal_id, new_cde_name in updates.items():
    cde_drupal_mapping_df.loc[
        cde_drupal_mapping_df["drupal_id"] == drupal_id, "cde"
    ] = new_cde_name
    print(f'CDE name for drupal_id {drupal_id} updated to "{new_cde_name}".')


# Debug, change id as needed
print("\nDebug")
print(cde_drupal_mapping_df.loc[cde_drupal_mapping_df["drupal_id"]=="6546"])


In [ ]:
adds = {
    "5361": ("supplemental", "Patient Health Questionnaire 8 - PHQ 8"),
    "5376": ("supplemental", "Patient Health Questionnaire 9 - PHQ 9 ")
}

for drupal_id, (type_value, cde_name) in adds.items():
    new_row = {
        "cde": cde_name,
        "type": type_value,
        "drupal_id": drupal_id
    }
    # Use DataFrame columns for correct order
    new_row_ordered = {col: new_row[col] for col in cde_drupal_mapping_df.columns}
    cde_drupal_mapping_df = pd.concat([
        cde_drupal_mapping_df,
        pd.DataFrame([new_row_ordered])
    ], ignore_index=True)
    print(f'Added row for drupal_id {drupal_id}: {new_row_ordered}')

print("\nDebug")
print(cde_drupal_mapping_df.loc[cde_drupal_mapping_df["drupal_id"] == "5361"])

# HEAL Basic dataset - full and collapsed versions

## Map studies and domains

In [ ]:
def get_domains(
    row, 
    domain_set, 
    collapse=False, 
    cde_drupal_df=None, 
    cde_quest_df=None, 
    pain_map=None, 
    qol_map=None
):
    """
    Extracts and maps HEAL health domains from structural checklist record targets.
    """

    hdp_id = row["HDP ID"]
    try:
        selections_dict = ast.literal_eval(row["VLMD: CDE selections"])
        cde_selections = selections_dict.keys()
    except (ValueError, SyntaxError):
        print(f"Issue with {hdp_id}: {row}")
        return [], []
    domains = set()
    all_domains_found = set()
    for cde_selection in cde_selections:
        cde_selection_id, cde_selection_name = cde_selection.split(" ", 1)
        df_match = cde_drupal_df[cde_drupal_df["drupal_id"] == cde_selection_id]
        domain_match = pd.DataFrame()
        if not df_match.empty:
            for _, r in df_match.iterrows():
                if r.type == "core":
                    domain_match = cde_quest_df[
                        (cde_quest_df["Name of Questionnaire"].str.strip() == r.cde.strip()) &
                        (cde_quest_df["Core or Supplemental"].str.contains("CORE"))
                        ]
                elif r.type == "supplemental":
                    domain_match = cde_quest_df[
                        (cde_quest_df["Name of Questionnaire"].str.strip() == r.cde.strip()) &
                        (cde_quest_df["Core or Supplemental"] == "Supplemental")
                        ]
                else:
                    print(f"Unknow cde type for '{cde_selection}' : type '{r.type}'")
            if not domain_match.empty:
                domain = domain_match.iloc[0]["HEAL Domain"].strip()
                all_domains_found.add(domain)
                if collapse:
                    canonical_domain = pain_map.get(domain, domain)
                    canonical_domain = qol_map.get(canonical_domain, canonical_domain)
                    if canonical_domain in domain_set:
                        domains.add(canonical_domain)
                else:
                    if domain in domain_set:
                        domains.add(domain)
            else:
                print(f"Failed to match cde: ({cde_selection})")
                print(f"Drupal data:")
                print(df_match)
                print("\n")
    return domains, all_domains_found

Processes the filtered dataset to extract, sort, and isolate target domains across both Full and Collapsed domains.
Create a list of ids <-> CDEs

In [ ]:
results_full = {}
results_collapsed = {}
skipped_studies_full = []
skipped_domains_full = {}
excluded_domains_per_study_full = {}      
skipped_studies_collapsed = []
skipped_domains_collapsed = {}
excluded_domains_per_study_collapsed = {} 

for _, row in filtered_checklist_df.iterrows():
    hdp_id = row["HDP ID"]
    
    # Full domains
    domains, all_found = get_domains(
        row, 
        heal_pain_domains_full, 
        collapse=False,
        cde_drupal_df=cde_drupal_mapping_df,
        cde_quest_df=cde_questionnaires_df,
        pain_map=pain_collapse_map,
        qol_map=satisfaction_qol_collapse_map
    )
    included = sorted(list(domains))
    excluded = [d for d in all_found if d not in heal_pain_domains_full]
    if included:
        results_full[hdp_id] = included
        if excluded:
            excluded_domains_per_study_full[hdp_id] = excluded
    else:
        skipped_studies_full.append(hdp_id)
        skipped_domains_full[hdp_id] = list(all_found) if all_found else ["No domains found"]

    # Collapsed domains
    domains_c, all_found_c = get_domains(
        row, 
        heal_pain_domains_collapsed, 
        collapse=True,
        cde_drupal_df=cde_drupal_mapping_df,
        cde_quest_df=cde_questionnaires_df,
        pain_map=pain_collapse_map,
        qol_map=satisfaction_qol_collapse_map
    )
    included_c = sorted(list(domains_c))
    mapped_found_c = set()
    for d in all_found_c:
        mapped_d = pain_collapse_map.get(d, d)
        mapped_d = satisfaction_qol_collapse_map.get(mapped_d, mapped_d)
        mapped_found_c.add(mapped_d)
    excluded_c = sorted([d for d in mapped_found_c if d not in heal_pain_domains_collapsed])
    
    if included_c:
        results_collapsed[hdp_id] = included_c
        if excluded_c:
            excluded_domains_per_study_collapsed[hdp_id] = excluded_c
    else:
        skipped_studies_collapsed.append(hdp_id)
        skipped_domains_collapsed[hdp_id] = sorted(list(mapped_found_c)) if mapped_found_c else ["No domains found"]

Investigate results (optional)

In [ ]:
print("Basic FULL results:")
for hdp_id in results_full.keys():
    included = results_full[hdp_id]
    excluded = excluded_domains_per_study_full.get(hdp_id, [])
    print(hdp_id)
    print("Domains Included:", included)
    print("Domains Excluded:", excluded)
    print("-" * 40)

print("\nBasic COLLAPSED results:")
for hdp_id in results_collapsed.keys():
    included = results_collapsed[hdp_id]
    excluded = excluded_domains_per_study_collapsed.get(hdp_id, [])
    print(hdp_id)
    print("Domains Included:", included)
    print("Domains Excluded:", excluded)
    print("-" * 40)

print("\nStudies excluded (FULL):")
for hdp_id in skipped_studies_full:
    domains = skipped_domains_full.get(hdp_id, [])
    print(f"{hdp_id}, Domains: {domains}")

print("\nStudies excluded (COLLAPSED):")
for hdp_id in skipped_studies_collapsed:
    domains = skipped_domains_collapsed.get(hdp_id, [])
    print(f"{hdp_id}, Domains: {domains}")

In [ ]:
print(f"{len(results_full)} studies with full domains; {len(results_collapsed)} with collapsed domains for CDE basic datasets")

## Get abstracts, prepare and save datasets

Get abstract and save datasets for ML

In [ ]:
def build_and_save(results, fname, exclude_ids, exclude_appl_ids, output_dir):
    """
    Queries the HEAL MDS API to compile structural metadata, filters benchmarks, 
    and saves target multi-label classification outputs to disk.
    """
    output_rows = []
    with requests.Session() as session:
        for hdp_id, domains in results.items():
            url = f"https://healdata.org/mds/metadata/{hdp_id}"
            try:
                r = requests.get(url)
                if r.status_code == 200:
                    content = r.json()
                    abstract_text = content.get("nih_reporter", {}).get("abstract_text", "")
                    appl_id = content.get("nih_reporter", {}).get("appl_id", "")
                    project_num = str(content.get("nih_reporter", {}).get("project_num", ""))
                else:
                    print(f"Issues with downloading data for {hdp_id}: {r.status_code}")
                    continue
            except Exception as e:
                print(f"Issues with downloading data for {hdp_id}: {e}")
                continue
            if hdp_id in exclude_ids or str(appl_id) in exclude_appl_ids:
                print(f"Filtered benchmark study detected ({hdp_id} / {appl_id}), skipping entry.")
                continue
            if abstract_text:
                output_rows.append({
                    "hdp_id": hdp_id,
                    "appl_id": appl_id,
                    "project_num": project_num,
                    "abstract_text": abstract_text,
                    "domains": ",".join(sorted(domains))
                })
            else:
                print(f"No abstract found in study {hdp_id}, skipping")
    df_out = pd.DataFrame(output_rows)
    output_path = os.path.join(output_dir, fname)
    df_out.to_csv(output_path, sep="\t", index=False)
    print(f"\nSaved {fname} (multi-label, one row per hdp_id)")
    return df_out


In [ ]:
heal_abstract_domain_basic_full = build_and_save(
    results=results_full, 
    fname="heal_abstract_domain_basic_full.tsv",
    exclude_ids=exclude_ids,
    exclude_appl_ids=exclude_appl_ids,
    output_dir=output_dir
)
heal_abstract_domain_basic_full

In [ ]:
heal_abstract_domain_basic_collapsed = build_and_save(
    results=results_collapsed, 
    fname="heal_abstract_domain_basic_collapsed.tsv",
    exclude_ids=exclude_ids,
    exclude_appl_ids=exclude_appl_ids,
    output_dir=output_dir
)
heal_abstract_domain_basic_collapsed

# HEAL Extended dataset - full and collapsed versions

This code fetches all HEAL studies, matches spending categories to domains using keyword mapping, merges with results (pain domains), excludes specified IDs, and saves only non-repeated domains for each HDP ID.

## Get all HEAL studies

In [ ]:
discovery_url = "https://healdata.org/mds/metadata?_guid_type=discovery_metadata&_guid_type=unregistered_discovery_metadata&data=True&limit=9999"
r = requests.get(discovery_url)
if r.status_code != 200:
    raise RuntimeError("Failed to fetch studies from healdata.org")
studies = r.json()

Investigate results (optional)

In [ ]:
# Print studies with no spending_categories, no abstract
no_sc_studies = []
yes_sc_studies = []
no_abstract_studies = []
yes_abstract_studies = []
all_spending_categories = set()

for hdp_id, study in studies.items():
    sc_text = study.get("nih_reporter", {}).get("spending_categories_desc", None)
    abstract_text = study.get("nih_reporter", {}).get("abstract_text", None)

    if sc_text is None or str(sc_text).strip() == "":
        no_sc_studies.append(hdp_id)
    else:
        yes_sc_studies.append(hdp_id)
        for cat in sc_text.split(";"):
            cat = cat.strip()
            if cat:
                all_spending_categories.add(cat)

    if abstract_text is None or str(abstract_text).strip() == "":
        no_abstract_studies.append(hdp_id)
    else:
        yes_abstract_studies.append(hdp_id)

print(f"Total studies: {len(studies)}")

print(f"\nStudies with abstract_text: {len(yes_abstract_studies)}")
print(f"Studies with NO abstract_text (empty or missing): {len(no_abstract_studies)}")
if no_abstract_studies:
    print("Example IDs (no abstract):", no_abstract_studies[:10])

print(f"\nStudies with spending_categories_desc: {len(yes_sc_studies)}")
print(f"Studies with NO spending_categories_desc (empty or missing): {len(no_sc_studies)}")
if no_sc_studies:
    print("Example IDs (no spending_categories):", no_sc_studies[:10])
print("\nAll spending_categories values across studies:")
for category in sorted(all_spending_categories):
    print("-", category)

## Map studies and domains

Create a spending category keywords mapping to domains

In [ ]:
# add all your domains/keywords here as needed
domain_keywords_full = {
    "Pain Intensity": ["pain", "intensity"],
    "Pain Interference": ["pain", "interference"],
    "Physical Functioning": ["physical","functioning"],
    "Sleep": ["sleep"],
    "Pain Catastrophizing": ["pain", "catastrophizing"],
    "Depression": ["depression"],
    "Anxiety": ["anxiety"],
    "Global Satisfaction with Treatment": ["satisfaction"],
    "Substance Use": ["substance", "use"],
    "Quality of Life": ["quality of life", "quality"]
}

In [ ]:
domain_keywords_collapsed = {}
for full_domain, keywords in domain_keywords_full.items():
    # Collapse mapping: pain and satisfaction
    if full_domain in pain_collapse_map:
        collapsed_domain = pain_collapse_map[full_domain]
    elif full_domain in satisfaction_qol_collapse_map:
        collapsed_domain = satisfaction_qol_collapse_map[full_domain]
    else:
        collapsed_domain = full_domain

    # Accumulate keywords for each collapsed domain
    if collapsed_domain not in domain_keywords_collapsed:
        domain_keywords_collapsed[collapsed_domain] = set()
    domain_keywords_collapsed[collapsed_domain].update(keywords)

# Convert sets to sorted lists for easy display/use
for domain in domain_keywords_collapsed:
    domain_keywords_collapsed[domain] = sorted(domain_keywords_collapsed[domain])

print("domain_keywords_collapsed:")
for domain, keywords in domain_keywords_collapsed.items():
    print(f"{domain}: {keywords}")

## Get abstracts, prepare and save datasets

In [ ]:
def build_extended(
    studies, 
    results, 
    domain_keywords, 
    domain_set, 
    exclude_ids, 
    exclude_appl_ids, 
    output_dir, 
    fname
):
    """
    Processes fetched HEAL discovery metadata, maps spending categories via keywords,
    resolves user-submitted labels, filters out benchmarks, and writes out a TSV file.
    """
    output_rows = []
    num_with_domains_abstracts = 0
    for hdp_id, study in studies.items():
        if not hdp_id:
            continue
        nih_reporter = study.get("nih_reporter", {})
        appl_id = str(nih_reporter.get("appl_id", ""))
        project_num = str(nih_reporter.get("project_num", ""))
        if hdp_id in exclude_ids or appl_id in exclude_appl_ids:
            continue
        domains = set()
        if hdp_id in results:
            domains.update(results[hdp_id])
        spending_categories_str = nih_reporter.get("spending_categories_desc", "") or ""
        spending_categories = [cat.strip() for cat in spending_categories_str.split(";") if cat.strip()]
        for category in spending_categories:
            cat_lower = category.lower()
            for domain, keywords in domain_keywords.items():
                if domain in domains:
                    continue
                for kw in keywords:
                    if kw in cat_lower:
                        domains.add(domain)
                        break
        domains = sorted([d for d in domains if d in domain_set])
        if domains:
            abstract_text = nih_reporter.get("abstract_text", "")
            if abstract_text:
                num_with_domains_abstracts += 1
                output_rows.append({
                    "hdp_id": hdp_id,
                    "appl_id": appl_id,
                    "project_num": project_num,
                    "abstract_text": abstract_text,
                    "domains": ",".join(domains)
                })
    df_out = pd.DataFrame(output_rows)
    output_path = os.path.join(output_dir, fname)
    df_out.to_csv(output_path, sep="\t", index=False)

    print(f"Saved {fname} with {num_with_domains_abstracts} studies.")
    return df_out

In [ ]:
heal_abstract_domain_extended_full = build_extended(
    studies=studies,
    results=results_full, 
    domain_keywords=domain_keywords_full, 
    domain_set=heal_pain_domains_full, 
    exclude_ids=exclude_ids,
    exclude_appl_ids=exclude_appl_ids,
    output_dir=output_dir,
    fname="heal_abstract_domain_extended_full.tsv"
)
heal_abstract_domain_extended_full

In [ ]:
heal_abstract_domain_extended_collapsed = build_extended(
    studies=studies,
    results=results_collapsed, 
    domain_keywords=domain_keywords_collapsed, 
    domain_set=heal_pain_domains_collapsed, 
    exclude_ids=exclude_ids,
    exclude_appl_ids=exclude_appl_ids,
    output_dir=output_dir,
    fname="heal_abstract_domain_extended_collapsed.tsv"
)
heal_abstract_domain_extended_collapsed

In [ ]:
# Union with basic
# --- FULL ---
basic_full_df = heal_abstract_domain_basic_full.copy()
extended_full_df = heal_abstract_domain_extended_full.copy()

missing_ids_full = set(basic_full_df["hdp_id"]) - set(extended_full_df["hdp_id"])
print("Basic_full IDs missing in extended_full:", sorted(missing_ids_full))

if missing_ids_full:
    missing_rows_full = basic_full_df[basic_full_df["hdp_id"].isin(missing_ids_full)]
    num_added = len(missing_rows_full)
    extended_full_df = pd.concat([extended_full_df, missing_rows_full], ignore_index=True)
    print(f"Added {num_added} studies from basic_full to extended_full.")

extended_full_df = extended_full_df.drop_duplicates(subset="hdp_id", keep="first")

extended_full_path = os.path.join(output_dir, "heal_abstract_domain_extended_full.tsv")
extended_full_df.to_csv(extended_full_path, sep="\t", index=False)
heal_abstract_domain_extended_full = extended_full_df

# --- COLLAPSED ---
basic_collapsed_df = heal_abstract_domain_basic_collapsed.copy()
extended_collapsed_df = heal_abstract_domain_extended_collapsed.copy()

missing_ids_collapsed = set(basic_collapsed_df["hdp_id"]) - set(extended_collapsed_df["hdp_id"])
print("Basic_collapsed IDs missing in extended_collapsed:", sorted(missing_ids_collapsed))

if missing_ids_collapsed:
    missing_rows_collapsed = basic_collapsed_df[basic_collapsed_df["hdp_id"].isin(missing_ids_collapsed)]
    num_added_collapsed = len(missing_rows_collapsed)
    extended_collapsed_df = pd.concat([extended_collapsed_df, missing_rows_collapsed], ignore_index=True)
    print(f"Added {num_added_collapsed} studies from basic_collapsed to extended_collapsed.")

extended_collapsed_df = extended_collapsed_df.drop_duplicates(subset="hdp_id", keep="first")

extended_collapsed_path = os.path.join(output_dir, "heal_abstract_domain_extended_collapsed.tsv")
extended_collapsed_df.to_csv(extended_collapsed_path, sep="\t", index=False)
heal_abstract_domain_extended_collapsed = extended_collapsed_df

# --- Summary for all HEAL datasets ---
print("\n--- All HEAL dataset sizes ---")
print(f"heal_abstract_domain_basic_full:         {len(heal_abstract_domain_basic_full)} studies")
print(f"heal_abstract_domain_basic_collapsed:    {len(heal_abstract_domain_basic_collapsed)} studies")
print(f"heal_abstract_domain_extended_full:      {len(heal_abstract_domain_extended_full)} studies")
print(f"heal_abstract_domain_extended_collapsed: {len(heal_abstract_domain_extended_collapsed)} studies")

# NIH Reporter dataset

## Helper functions

In [ ]:
def get_nih_reporter_projects(output_dir, fiscal_years, nih_agencies):
    """
    Checks for locally processed data, falls back to raw data parsing, or queries 
    the external NIH RePORTER API downstream before deduplicating rows on project_num.
    """
    projects_jsonl_path = os.path.join(output_dir, "nih-reporter_projects.jsonl")
    processed_jsonl_path = os.path.join(output_dir, "nih-reporter_projects_processed.jsonl")
    
    # CASE 1: Processed file exists
    if os.path.exists(processed_jsonl_path):
        print(f"[CASE 1] Loading processed projects from {processed_jsonl_path}...")
        df = pd.read_json(processed_jsonl_path, lines=True)
        print(f"Loaded {len(df)} projects from file.")
        return df.to_dict(orient='records')
        
    # CASE 2: Only raw file exists, filter and deduplicate
    elif os.path.exists(projects_jsonl_path):
        print(f"[CASE 2] Processed cache not found. Filtering and deduplicating raw file...")
        df = pd.read_json(projects_jsonl_path, lines=True)
        mask = (
            df['project_num'].notnull() & df['project_num'].astype(str).str.strip().ne('') &
            df['abstract_text'].notnull() & df['abstract_text'].astype(str).str.strip().ne('') &
            df['spending_categories_desc'].notnull() & df['spending_categories_desc'].astype(str).str.strip().ne('')
        )
        df = df[mask].drop_duplicates(subset='project_num')
        df.to_json(processed_jsonl_path, orient='records', lines=True)
        print(f"Wrote {len(df)} projects to {processed_jsonl_path}")
        return df.to_dict(orient='records')
        
    # CASE 3: No files exist, run the NIH API query, then filter and deduplicate
    else:
        print("[CASE 3] No project cache files found. Fetching from remote NIH Reporter API...")
        count = 0
        with open(projects_jsonl_path, "w") as f:
            for year in fiscal_years:
                for agency in nih_agencies:
                    offset = 0
                    batch = 0
                    more = True
                    while more:
                        if offset > 14999:
                            print(f"Year {year}, Agency {agency} exceeded offset limit, skipped.")
                            break
                        payload = {
                            "criteria": {"fiscal_years": [year], "agencies": [agency]},
                            "include_fields": ["ApplId", "ProjectNum", "AbstractText", "SpendingCategoriesDesc"],
                            "offset": offset,
                            "limit": 500
                        }
                        try:
                            r = requests.post("https://api.reporter.nih.gov/v2/projects/search", json=payload, timeout=15)
                            if r.status_code != 200:
                                print(f"API Error ({r.status_code}) for Year {year}, Agency {agency}. Skipping batch.")
                                break
                            res = r.json()
                        except Exception as e:
                            print(f"Network error on API payload fetch: {e}")
                            break
                            
                        results = res.get("results", [])
                        print(f"Year {year}, Agency {agency}, Offset {offset}: {len(results)} projects...")
                        
                        for proj in results:
                            if proj:
                                json.dump(proj, f)
                                f.write("\n")
                                count += 1
                                
                        if len(results) < payload["limit"]:
                            more = False
                        offset += payload["limit"]
                        batch += 1
                        
        print(f"Finished writing {count} raw elements to disk. Ingesting parsing layer...")
        df = pd.read_json(projects_jsonl_path, lines=True)
        mask = (
            df['project_num'].notnull() & df['project_num'].astype(str).str.strip().ne('') &
            df['abstract_text'].notnull() & df['abstract_text'].astype(str).str.strip().ne('') &
            df['spending_categories_desc'].notnull() & df['spending_categories_desc'].astype(str).str.strip().ne('')
        )
        df = df[mask].drop_duplicates(subset='project_num')
        df.to_json(processed_jsonl_path, orient='records', lines=True)
        print(f"Wrote {len(df)} finalized entries to cache schema.")
        return df.to_dict(orient='records')

In [ ]:
def build_nih_reporter_dataset(projects, domain_keywords, appl_id_to_hdp_id, output_dir, fname):
    """
    Parses and sanitizes the collected abstract text strings, correlates keywords against 
    spending category descriptors, maps linked HDP IDs, and serializes records out to a clean TSV.
    """
    output_rows = []
    for proj in projects:
        pn = proj.get('project_num', '')
        appl_id = str(proj.get('appl_id', ''))
        abstract = proj.get('abstract_text', '')
        
        if not abstract:
            continue
            
        # Clean inline return breaks to secure structural one-line TSV cells
        abstract = abstract.replace('\r\n', '\\n').replace('\r', '\\n').replace('\n', '\\n').strip()
        sc_desc = proj.get('spending_categories_desc', "")
        sc_list = [cat.lower().strip() for cat in sc_desc.split(';') if cat.strip()]
        
        matched_domains = set()
        for domain, keywords in domain_keywords.items():
            for kw in keywords:
                if any(kw.lower() in cat for cat in sc_list):
                    matched_domains.add(domain)
                    break
                    
        if matched_domains:
            hdp_id = appl_id_to_hdp_id.get(appl_id, None)
            output_rows.append({
                "hdp_id": hdp_id,
                "project_num": pn,
                "appl_id": appl_id,
                "abstract_text": abstract,
                "domains": ",".join(sorted(matched_domains))
            })
            
    df_out = pd.DataFrame(output_rows)
    output_path = os.path.join(output_dir, fname)
    df_out.to_csv(
        output_path,
        sep='\t',
        index=False,
        quoting=csv.QUOTE_MINIMAL,
        escapechar='\\'
    )
    print(f"Saved dataset variant: {output_path} | Size: {len(df_out)} rows")
    return df_out

In [ ]:
def merge_domains_with_heal(nih_df, heal_df, key="hdp_id"):
    """
    Helper function combining label columns.
    """
    nih_df = nih_df.copy()
    heal_domains_map = heal_df.dropna(subset=[key]).set_index(key)["domains"].to_dict()
    for idx, row in nih_df.iterrows():
        k = row[key]
        if pd.isna(k) or k not in heal_domains_map:
            continue
        nih_domains = set(str(row["domains"]).split(",")) if pd.notna(row["domains"]) else set()
        heal_domains = set(str(heal_domains_map.get(k, "")).split(","))
        all_domains = ",".join(sorted(nih_domains | heal_domains))
        nih_df.at[idx, "domains"] = all_domains
    return nih_df

In [ ]:
def finalize_nih_reporter_union(nih_df, basic_df, extended_df, output_dir, fname):
    """
    Executes union masks bringing omitted basic/extended studies forward,
    merges overlapping categorization tags and saves to disk
    """
    nih_df = nih_df.copy()
    
    nih_hdp_ids = set(nih_df["hdp_id"].dropna())
    nih_appl_ids = set(nih_df["appl_id"].astype(str).dropna())
    
    # Isolate records in basic study tables missing from the main RePORTER extraction matrix
    basic_missing_mask = (
        ~basic_df["hdp_id"].isin(nih_hdp_ids) &
        ~basic_df["appl_id"].astype(str).isin(nih_appl_ids)
    )
    missing_basic_rows = basic_df[basic_missing_mask]
    if len(missing_basic_rows) > 0:
        nih_df = pd.concat([nih_df, missing_basic_rows], ignore_index=True)
        print(f"Union Extension: Added {len(missing_basic_rows)} missing studies from basic source.")
        
    # Isolate records in extended files missing from the main matrix
    extended_missing_mask = (
        ~extended_df["hdp_id"].isin(nih_hdp_ids) &
        ~extended_df["appl_id"].astype(str).isin(nih_appl_ids)
    )
    missing_extended_rows = extended_df[extended_missing_mask]
    if len(missing_extended_rows) > 0:
        nih_df = pd.concat([nih_df, missing_extended_rows], ignore_index=True)
        print(f"Union Extension: Added {len(missing_extended_rows)} missing studies from extended source.")
        
    # Coalesce multi-label classifications cleanly
    nih_df = merge_domains_with_heal(nih_df, basic_df, key="hdp_id")
    nih_df = merge_domains_with_heal(nih_df, extended_df, key="hdp_id")
    
    # Complete unique key assertions
    nih_df = nih_df.drop_duplicates(subset=("hdp_id", "appl_id", "project_num"), keep="first")
    
    # Save
    output_path = os.path.join(output_dir, fname)
    nih_df.to_csv(output_path, sep='\t', index=False, quoting=csv.QUOTE_MINIMAL, escapechar='\\')
    print(f"Completed and stored final data array: {output_path} | Size: {len(nih_df)}")
    return nih_df

## Create datasets

Define years and agencies

In [ ]:
fiscal_years = list(range(1984, 2027))
nih_agencies = [
    "NCI", "NIAID", "NHLBI", "NIDDK", "NINDS", "NIMH", "NIA", "NICHD",
    "NIGMS", "NEI", "NIEHS", "NIDCR", "NIDA", "NIAAA", "NINR", "HG",
    "NIBIB", "NIMHD", "NIDCD", "NCCIH", "NCATS", "LM", "CIT", "CC",
    "OD", "FIC", "CSR"
]

Get projects from NIH Reporter

This step might be memory intensive. If you experience out-of-memory issues, try to adjust fiscal_years and nih_agencies to decrease memory used in processing.

In [ ]:
projects = get_nih_reporter_projects(output_dir=output_dir, fiscal_years=fiscal_years, nih_agencies=nih_agencies)

Create a map connecting appl_id and hdp_id for existing HEAL studies in the NIH Reporter studies

In [ ]:
appl_id_to_hdp_id = {}
for idx, row in heal_abstract_domain_extended_collapsed.iterrows():
    if pd.notna(row["appl_id"]):
        appl_id_to_hdp_id[str(row["appl_id"])] = row["hdp_id"]

Full dataset

In [ ]:
nih_reporter_full_base = build_nih_reporter_dataset(
    projects=projects,
    domain_keywords=domain_keywords_full,
    appl_id_to_hdp_id=appl_id_to_hdp_id,
    output_dir=output_dir,
    fname="nih-reporter_abstract_domain_full.tsv"
)

Optionally combine with HEAL studies - labels

In [ ]:
nih_reporter_full_df = finalize_nih_reporter_union(
    nih_df=nih_reporter_full_base,
    basic_df=heal_abstract_domain_basic_full,
    extended_df=heal_abstract_domain_extended_full,
    output_dir=output_dir,
    fname="nih-reporter_abstract_domain_full.tsv"
)

Collapsed dataset

In [ ]:
nih_reporter_collapsed_base = build_nih_reporter_dataset(
    projects=projects,
    domain_keywords=domain_keywords_collapsed,
    appl_id_to_hdp_id=appl_id_to_hdp_id,
    output_dir=output_dir,
    fname="nih-reporter_abstract_domain_collapsed.tsv"
)

Optionally combine with HEAL studies - labels

In [ ]:
nih_reporter_collapsed_df = finalize_nih_reporter_union(
    nih_df=nih_reporter_collapsed_base,
    basic_df=heal_abstract_domain_basic_collapsed,
    extended_df=heal_abstract_domain_extended_collapsed,
    output_dir=output_dir,
    fname="nih-reporter_abstract_domain_collapsed.tsv"
)


# Sanity check

These checks load datasets from the disk and can be run independetly from previous code

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Update as needed
plots_dir = "./outputs/plots"
datasets_dir = "./outputs/datasets"

In [ ]:
# Update as needed
datasets_files = {
    "heal-basic_full": "heal_abstract_domain_basic_full.tsv",
    "heal-extended_full": "heal_abstract_domain_extended_full.tsv",
    "nih-reporter_full": "nih-reporter_abstract_domain_full.tsv",
    "heal-basic_collapsed": "heal_abstract_domain_basic_collapsed.tsv",
    "heal-extended_collapsed": "heal_abstract_domain_extended_collapsed.tsv",
    "nih-reporter_collapsed": "nih-reporter_abstract_domain_collapsed.tsv",
}

In [ ]:
datasets = []
for label_id, file_name in datasets_files.items():
    absolute_tsv_path = os.path.join(datasets_dir, file_name)
    
    if os.path.exists(absolute_tsv_path):
        df_loaded = pd.read_csv(absolute_tsv_path, sep="\t")
        datasets.append((label_id, df_loaded))
        print(f"Loaded '{label_id}' -> Shape: {df_loaded.shape}")
    else:
        print(f"❌ File missing, verify system mount path: {absolute_tsv_path}")

# Separate data sets into isolated operational collection structures
full_names = ["heal-basic_full", "heal-extended_full", "nih-reporter_full"]
full_datasets = [(name, df) for name, df in datasets if name in full_names]

collapsed_names = ["heal-basic_collapsed", "heal-extended_collapsed", "nih-reporter_collapsed"]
collapsed_datasets = [(name, df) for name, df in datasets if name in collapsed_names]

# Summary structural verification print-outs
print(f"\nConfiguration summary: {len(full_datasets)} Full / {len(collapsed_datasets)} Collapsed tracking lists initialized.")


Check unique domains

In [ ]:
for dataset_name, dataset in datasets:
    print(f"Unique domains in {dataset_name}:")
    print(sorted({d for ds in dataset["domains"].str.split(",") for d in ds}))

Check domains distribution accross datasets

In [ ]:
full_domains = sorted({
    d
    for _, df in full_datasets
    for ds in df["domains"].str.split(",")
    for d in ds
})

full_counts = {}
for name, df in full_datasets:
    domain_counter = {domain: 0 for domain in full_domains}
    for domains in df["domains"].str.split(","):
        for d in domains:
            domain_counter[d] += 1
    full_counts[name] = domain_counter

full_summary_df = pd.DataFrame(full_counts).T[full_domains].astype(int)
display(full_summary_df)

In [ ]:
collapsed_names = ["heal-basic_collapsed", "heal-extended_collapsed", "nih-reporter_collapsed"]
collapsed_datasets = [(name, df) for name, df in datasets if name in collapsed_names]

collapsed_domains = sorted({
    d
    for _, df in collapsed_datasets
    for ds in df["domains"].str.split(",")
    for d in ds
})

collapsed_counts = {}
for name, df in collapsed_datasets:
    domain_counter = {domain: 0 for domain in collapsed_domains}
    for domains in df["domains"].str.split(","):
        for d in domains:
            domain_counter[d] += 1
    collapsed_counts[name] = domain_counter

collapsed_summary_df = pd.DataFrame(collapsed_counts).T[collapsed_domains].astype(int)
display(collapsed_summary_df)

In [ ]:
# --- FULL domain populations ---
fig, axes = plt.subplots(len(full_datasets), 1, figsize=(10, 3*len(full_datasets)), sharex=True)
for i, (name, _) in enumerate(full_datasets):
    dataset, domain_type = name.split("_")   
    values = full_summary_df.loc[name]
    bar_container = axes[i].bar(full_summary_df.columns, full_summary_df.loc[name], color='deepskyblue')
    axes[i].set_ylabel("# studies")
    axes[i].set_title(f"{dataset} dataset, {domain_type} domains")
    axes[i].grid(axis="y", linestyle=":", alpha=0.5)
    axes[i].bar_label(bar_container, label_type="edge", padding=3)
    axes[i].set_ylim(0, max(values)*1.15)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

full_plot_path = os.path.join(plots_dir, "full_domain_populations.png")
plt.savefig(full_plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved FULL domain populations plot to: {full_plot_path}")

# --- COLLAPSED domain populations ---
fig, axes = plt.subplots(len(collapsed_datasets), 1, figsize=(10, 3*len(collapsed_datasets)), sharex=True)
for i, (name, _) in enumerate(collapsed_datasets):
    dataset, domain_type = name.split("_")
    values = collapsed_summary_df.loc[name]
    bar_container = axes[i].bar(collapsed_summary_df.columns, collapsed_summary_df.loc[name], color='lightcoral')
    axes[i].set_ylabel("# studies")
    axes[i].set_title(f"{dataset} dataset, {domain_type} domains")
    axes[i].grid(axis="y", linestyle=":", alpha=0.5)
    axes[i].bar_label(bar_container, label_type="edge", padding=3)
    axes[i].set_ylim(0, max(values)*1.15)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

collapsed_plot_path = os.path.join(plots_dir, "collapsed_domain_populations.png")
plt.savefig(collapsed_plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved COLLAPSED domain populations plot to: {collapsed_plot_path}")